# 조건 드롭아웃 실험: 원곡 1장 암기 줄이기

기존 모델은 (장, 박자, 장 안 위치) 조합만 보고 원곡 1장·4장을 그대로 외워 냅니다.
이 Notebook은 같은 설정·같은 Seed로 두 모델을 차례로 학습하고 비교합니다.

| 모델 | 설정 파일 | 차이 |
|---|---|---|
| baseline | `configs/train_maskgit.yaml` | 기존과 같음 |
| conddrop | `configs/train_maskgit_conddrop.yaml` | 학습 중 마디마다 50% 확률로 장·위치 조건을 '알 수 없음'으로 바꿈 |

**실행 순서**: Runtime → Change runtime type에서 **GPU**를 고른 뒤 위에서 아래로 실행합니다.
실행 방법 전체는 `docs/condition_dropout_colab.md`에 있습니다.

**덮어쓰기 없음**: 결과 파일 이름에 `RUN_TAG`가 붙고, 같은 이름이 이미 있으면 학습하지 않고 멈춥니다.
기존 `DiffDodree` clone은 건드리지 않고 `DiffDodree_conddrop`이라는 별도 clone을 씁니다.

In [ ]:
# Cell 0 — 실험 전용 clone 준비 (Colab에서만 동작합니다)
# 기존 MyDrive/DiffDodree는 건드리지 않고, 실험 branch를 별도 폴더에 clone합니다.
from pathlib import Path
import importlib.util
import os
import subprocess

REPO_URL = "https://github.com/SHlee-TDA/DiffDodree.git"
BRANCH = "claude/generated-song-verse-1-4-identity-182b42"
EXPERIMENT_ROOT = Path("/content/drive/MyDrive/DiffDodree_conddrop")

try:
    _in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    _in_colab = False

if _in_colab:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

    def git(*args: str) -> str:
        return subprocess.check_output(["git", "-C", str(EXPERIMENT_ROOT), *args], text=True).strip()

    if not EXPERIMENT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(EXPERIMENT_ROOT)],
            check=True,
        )
    else:
        if git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError(f"{EXPERIMENT_ROOT}는 다른 저장소의 clone입니다.")
        if git("status", "--porcelain", "--untracked-files=no"):
            raise RuntimeError(
                f"{EXPERIMENT_ROOT}에 commit하지 않은 수정이 있습니다. 덮어쓰지 않도록 여기서 멈춥니다."
            )
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(EXPERIMENT_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(EXPERIMENT_ROOT), "merge", "--ff-only", f"origin/{BRANCH}"], check=True
        )
    print("Commit:", git("log", "-1", "--oneline"))
    os.environ["GUGAKGEN_PROJECT_ROOT"] = str(EXPERIMENT_ROOT)
else:
    print("Colab이 아니므로 현재 project folder를 그대로 사용합니다.")

In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

In [ ]:
# 실험 code가 들어 있는 commit인지, GPU가 잡혔는지 확인합니다.
import subprocess

import torch

from gugakgen.workflows import conddrop_experiment  # 없으면 branch가 잘못된 것입니다.

print("Commit:", subprocess.check_output(["git", "-C", str(project_root), "log", "-1", "--oneline"], text=True).strip())
print("Runtime:", runtime.as_dict())
if runtime.backend == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 연구자가 주로 바꾸는 값입니다.
from datetime import datetime, timezone, timedelta

RUN_MODE = "full"  # 코드 경로만 확인하려면 "smoke" (3 step)
STEPS = 6000       # 두 모델에 같은 값을 씁니다.
EVAL_SEEDS = list(range(260807, 260867))  # 60개, 이전 비교와 같은 Seed

# None이면 실행 시각으로 새 이름을 만듭니다.
# 연결이 끊겨 다시 실행할 때는 이전 실행에서 출력된 RUN_TAG를 그대로 넣으세요.
# 이미 끝난 모델은 건너뛰고, 남은 모델만 학습한 뒤 평가합니다.
RUN_TAG = None

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE는 'smoke' 또는 'full'이어야 합니다.")
if RUN_MODE == "full" and runtime.backend != "cuda":
    raise RuntimeError(
        "full 학습은 CUDA GPU에서만 실행합니다. Runtime → Change runtime type에서 GPU를 고르세요. "
        "(메모리 8GB MacBook에서 두 모델을 돌리면 스왑으로 사실상 멈춥니다.)"
    )
if RUN_MODE == "smoke":
    EVAL_SEEDS = EVAL_SEEDS[:2]

In [ ]:
# 두 모델의 runtime configuration을 씁니다. 결과 이름이 이미 있으면 덮어쓰지 않고 멈춥니다.
from gugakgen.config import load_yaml
from gugakgen.workflows.conddrop_experiment import BASE_CONFIGS, write_runtime_configs

RUNTIME_CONFIG_DIR = project_root / "artifacts/runtime_configs"


def tagged_config_path(name: str, tag: str):
    base = load_yaml(project_root / BASE_CONFIGS[name])
    return RUNTIME_CONFIG_DIR / f"{base['experiment_name']}_{tag}.yaml"


if RUN_TAG is None:
    RUN_TAG = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d_%H%M")
    config_paths = write_runtime_configs(project_root, run_tag=RUN_TAG, steps=STEPS)
else:
    config_paths = {name: tagged_config_path(name, RUN_TAG) for name in BASE_CONFIGS}
    missing = [str(path) for path in config_paths.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"RUN_TAG {RUN_TAG!r}의 configuration이 없습니다: {missing}")


def checkpoint_path(config_path):
    config = load_yaml(config_path)
    name = Path(config["artifacts"]["checkpoint_name"])
    if RUN_MODE != "full":
        name = Path(f"{name.stem}_smoke{name.suffix}")
    return project_root / config["artifacts"]["output_dir"] / name


checkpoints = {name: checkpoint_path(path) for name, path in config_paths.items()}
print("RUN_TAG =", repr(RUN_TAG), " ← 다시 실행할 때 이 값을 쓰세요.")
for name in config_paths:
    state = "이미 있음 → 학습 건너뜀" if checkpoints[name].exists() else "학습 예정"
    print(f"- {name}: {checkpoints[name].relative_to(project_root)} ({state})")

In [ ]:
# 두 모델을 한 번에 하나씩 학습합니다. 진행 표시가 없으니 셀이 끝날 때까지 기다리세요.
# 학습 중 Checkpoint는 runtime의 임시 폴더에 쓰이고, 끝난 뒤에 Drive로 복사됩니다.
# 따라서 학습 도중 연결이 끊기면 그 모델은 처음부터 다시 학습해야 합니다.
import json
import time

from gugakgen.workflows.train import train_from_config

histories = {}
for name, config_path in config_paths.items():
    if checkpoints[name].exists():
        print(f"{name}: 이미 학습됨, 건너뜁니다.")
    else:
        started = time.time()
        print(f"{name}: 학습 시작 ({STEPS if RUN_MODE == 'full' else 'smoke'} step)", flush=True)
        result = train_from_config(config_path, project_root=project_root, run_mode=RUN_MODE, runtime=runtime)
        print(f"{name}: 완료 {(time.time() - started) / 60:.1f}분, SHA-256 {result.checkpoint_sha256[:12]}")
    config = load_yaml(config_path)
    history_name = Path(config["artifacts"]["history_name"])
    if RUN_MODE != "full":
        history_name = Path(f"{history_name.stem}_smoke{history_name.suffix}")
    history_path = project_root / config["artifacts"]["output_dir"] / history_name
    histories[name] = json.loads(history_path.read_text(encoding="utf-8"))["history"]

In [ ]:
# 학습 중 '암기 정도'의 변화입니다.
# recall_keyed: 모든 음을 가리고 장·위치만 줬을 때 원곡 마디를 그대로 맞히는 비율
# recall_null : 장·위치까지 '알 수 없음'으로 줬을 때의 비율 (conddrop만 해당)
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

fig, ax = plt.subplots(figsize=(8, 4))
for name, history in histories.items():
    frame = pd.DataFrame(history)
    for column in ("recall_keyed", "recall_null"):
        if column in frame:
            ax.plot(frame["step"], frame[column], marker="o", label=f"{name} · {column}")
ax.set_xlabel("step")
ax.set_ylabel("recall of original measures")
ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.2)
ax.legend()
plt.show()

In [ ]:
# 두 모델을 같은 Seed로 생성·평가합니다. 결과 JSON은 이름이 겹치면 저장하지 않고 멈춥니다.
from gugakgen.workflows.conddrop_experiment import evaluate_checkpoint, summary_rows

evaluation_dir = project_root / "artifacts/evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)
report_path = evaluation_dir / f"conddrop_{RUN_TAG}{'' if RUN_MODE == 'full' else '_smoke'}.json"
if report_path.exists():
    raise FileExistsError(f"평가 결과가 이미 있습니다: {report_path}")

reports = {}
for name, config_path in config_paths.items():
    print(f"{name}: 평가 중 ({len(EVAL_SEEDS)} Seed)", flush=True)
    reports[name] = evaluate_checkpoint(
        checkpoints[name], config_path, project_root=project_root, runtime=runtime, seeds=EVAL_SEEDS
    )

report_path.write_text(
    json.dumps({"run_tag": RUN_TAG, "run_mode": RUN_MODE, "steps": STEPS, "seeds": EVAL_SEEDS, "reports": reports},
               indent=1, ensure_ascii=False),
    encoding="utf-8",
)
print("저장:", report_path.relative_to(project_root))

In [ ]:
# 결과 표
recall_table = pd.DataFrame({name: report["recall"] for name, report in reports.items()}).T
sampling_rows, two_stage_rows = summary_rows(reports)

print("1) 암기 정도 (모든 음을 가린 입력에서 원곡 마디를 그대로 맞히는 비율)")
display(recall_table)

print("2) 생성 결과 — setting은 장·위치를 '알 수 없음'으로 준 마디 (keyed=없음, null_ch1=1장, null_ch1_ch4=1·4장, null_all=전체)")
display(pd.DataFrame(sampling_rows).set_index(["model", "setting"]))

print("3) 2단계 환입 실험 — 새로 만든 1장이 4장에서 돌아오는가 (true=1→4 관계, none=관계 없음, negative=2→4 대조)")
display(pd.DataFrame(two_stage_rows).set_index(["model", "relation"]))

## 표 읽는 법

**1) 암기 정도**
- baseline의 `recall_keyed`가 높고, conddrop의 `recall_null`이 낮으면 조건 드롭아웃이 '주소로 외우기'를 끊은 것입니다.

**2) 생성 결과**
- `ch1_all_original`: 60곡 중 1장 네 마디가 모두 원곡과 같은 곡 수
- `ch1_novel_mean`: 1장 네 마디 중 원곡 어디에도 없는 마디 수의 평균
- `novel_returned_total`: 새로 만든 1장 마디가 4장에서 같은 자리로 다시 나온 횟수
- `transition_validity`, `pitch_jsd`: 형식이 깨지지 않았는지 보는 최소 점검입니다. 음악적 품질은 들어 보고 판단해야 합니다.

**3) 2단계 환입 실험**
- `return_rate_novel`이 핵심입니다. 1장이 새 마디일 때 4장이 그것을 따라가는 비율입니다.
- `true`가 `none`·`negative`보다 뚜렷하게 높아야 '환입 관계를 배웠다'고 말할 수 있습니다.

결과 JSON은 Drive의 `DiffDodree_conddrop/artifacts/evaluation/`에 남습니다. 이 경로를 알려 주시면 Mac에서 읽어 해석하겠습니다.